In [ ]:
"""
English Premier League (EPL) Historical Data Cleaning Pipeline
================================================================
Ingests 20+ years of scraped EPL CSV files (schema varies between older
and newer seasons), unifies them into a single clean DataFrame, and
applies football-domain-aware validation and null-handling rules.

Run this cell-by-cell in Google Colab. When prompted, select all CSV
files you want to merge (you can multi-select in the file dialog).
"""

import io
import os
import re
import zipfile
import glob
import numpy as np
import pandas as pd
from google.colab import files

pd.set_option("display.max_columns", None)

# ---------------------------------------------------------------------------
# INGESTION MODE
# ---------------------------------------------------------------------------
# A browser file picker (google.colab.files.upload) cannot select a folder
# directly -- that's a browser/OS limitation, not a Colab one. So we support
# two ways of handing over "a folder full of CSVs":
#
#   "zip"   -> Zip the folder on your machine first, upload the single .zip,
#              and this script extracts every .csv inside it (including in
#              subfolders). Simplest if the data is local.
#
#   "drive" -> Mount Google Drive and point DRIVE_FOLDER_PATH at the folder.
#              No upload needed at all. Best if the folder already lives in
#              Drive or you don't want to zip anything.
#
# Set INGEST_MODE to one of "zip" or "drive" before running.
INGEST_MODE = "zip"  # change to "drive" to use Google Drive instead

# Only used when INGEST_MODE == "drive". Path to the folder of CSVs inside
# your Drive, e.g. "/content/drive/MyDrive/epl_data".
DRIVE_FOLDER_PATH = "/content/drive/MyDrive/epl_data"


# ---------------------------------------------------------------------------
# PHASE 1: UNIFICATION & DATES
# ---------------------------------------------------------------------------

def _read_csvs_from_paths(csv_paths: list) -> pd.DataFrame:
    """Shared loader: read a list of on-disk CSV paths and concat them."""
    frames = []
    for path in csv_paths:
        try:
            df = pd.read_csv(path)
        except Exception as exc:
            print(f"  [SKIPPED] Could not parse '{path}': {exc}")
            continue

        df["__source_file"] = os.path.basename(path)  # provenance for debugging
        frames.append(df)
        print(f"  [LOADED] {os.path.basename(path)}: {len(df)} rows, "
              f"{len(df.columns)} cols")

    if not frames:
        raise ValueError("No valid CSV files were found/parsed.")

    # sort=False preserves column order of first appearance and performs
    # the outer-join union of all distinct columns across old/new schemas.
    # Older-season files lack HTHG/HTAG/HTR; concat fills those with NaN
    # for those rows rather than dropping any data ("structural nulls").
    unified = pd.concat(frames, ignore_index=True, sort=False)
    print(f"\nUnified dataset: {len(unified)} rows, {len(unified.columns)} cols")
    return unified


def upload_zip_and_concat_csvs() -> pd.DataFrame:
    """
    Upload a single .zip containing a folder of CSVs (nested subfolders
    are fine) and concatenate every .csv found inside it.
    """
    uploaded = files.upload()  # dict: {filename: bytes}
    zip_items = [(name, data) for name, data in uploaded.items()
                 if name.lower().endswith(".zip")]

    if not zip_items:
        raise ValueError("No .zip file was uploaded. Zip your folder of "
                          "CSVs and upload that single file.")

    extract_dir = "/content/epl_csv_extracted"
    os.makedirs(extract_dir, exist_ok=True)

    for zip_name, zip_bytes in zip_items:
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            zf.extractall(extract_dir)
        print(f"  [EXTRACTED] {zip_name} -> {extract_dir}")

    # Recurse into any subfolders inside the zip.
    csv_paths = glob.glob(os.path.join(extract_dir, "**", "*.csv"), recursive=True)
    if not csv_paths:
        raise ValueError(f"No .csv files found after extracting into "
                          f"{extract_dir}. Check the zip contents.")

    print(f"  Found {len(csv_paths)} CSV file(s) inside the zip.")
    return _read_csvs_from_paths(csv_paths)


def load_from_drive_folder(folder_path: str = DRIVE_FOLDER_PATH) -> pd.DataFrame:
    """
    Mount Google Drive (if not already mounted) and load every .csv found
    in folder_path (including subfolders). No upload step required.
    """
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

    if not os.path.isdir(folder_path):
        raise ValueError(f"Drive folder not found: {folder_path}. Update "
                          f"DRIVE_FOLDER_PATH to point at your CSV folder.")

    csv_paths = glob.glob(os.path.join(folder_path, "**", "*.csv"), recursive=True)
    if not csv_paths:
        raise ValueError(f"No .csv files found in {folder_path}.")

    print(f"  Found {len(csv_paths)} CSV file(s) in Drive folder.")
    return _read_csvs_from_paths(csv_paths)


def upload_and_concat_csvs() -> pd.DataFrame:
    """
    Entry point that dispatches to the configured ingestion mode
    (INGEST_MODE = "zip" or "drive"). See the comment block above for
    details on each mode.
    """
    if INGEST_MODE == "zip":
        return upload_zip_and_concat_csvs()
    elif INGEST_MODE == "drive":
        return load_from_drive_folder()
    else:
        raise ValueError(f"Unknown INGEST_MODE: {INGEST_MODE!r}. "
                          f"Use 'zip' or 'drive'.")


def parse_match_dates(df: pd.DataFrame, date_col: str = "Date") -> pd.DataFrame:
    """
    Parse the Date column into real datetimes.

    Web-scraped EPL data is notorious for inconsistent date formats across
    seasons/sources: 'dd/mm/yy', 'dd/mm/yyyy', occasionally ISO-like
    strings. We try a dayfirst parse first (standard for UK football data
    sources like football-data.co.uk), then fall back to a flexible parse
    for anything that didn't match, rather than failing the whole column.
    """
    df = df.copy()

    # Primary pass: UK-style day-first dates (most common for EPL scrapes).
    parsed = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")

    # Fallback pass: let pandas infer format for anything still unparsed
    # (e.g. rows that happened to come in a different string format).
    still_missing = parsed.isna() & df[date_col].notna()
    if still_missing.any():
        fallback = pd.to_datetime(
            df.loc[still_missing, date_col], errors="coerce", format="mixed"
        )
        parsed.loc[still_missing] = fallback

    df[date_col] = parsed

    unparseable = df[date_col].isna().sum()
    if unparseable:
        print(f"  [WARN] {unparseable} rows have an unparseable Date and "
              f"will have no Season assigned (will be caught by goal-null "
              f"drop logic later if also missing scores).")

    return df


def assign_season(df: pd.DataFrame, date_col: str = "Date") -> pd.DataFrame:
    """
    Derive a 'Season' label like '2007/2008' from the match date.

    Football logic: the EPL season runs August -> May. So:
      - A match in Aug-Dec of year Y belongs to season "Y/Y+1".
      - A match in Jan-Jul of year Y belongs to season "Y-1/Y".
    """
    df = df.copy()

    def _season_label(ts: pd.Timestamp):
        if pd.isna(ts):
            return np.nan
        year = ts.year
        if ts.month >= 8:  # August onward -> start of a new season
            start_year = year
        else:  # Jan-Jul -> still inside the season that started last year
            start_year = year - 1
        return f"{start_year}/{start_year + 1}"

    df["Season"] = df[date_col].apply(_season_label)
    return df


# ---------------------------------------------------------------------------
# PHASE 2: ENTITY RESOLUTION (TEAM NAME STANDARDIZATION)
# ---------------------------------------------------------------------------

# Canonical name mapping for commonly inconsistent EPL team name variants
# as they appear across different scrape sources/seasons. Extend this dict
# as new variants are discovered in incoming data.
TEAM_NAME_MAP = {
    "Man Utd": "Manchester United",
    "Man United": "Manchester United",
    "Manchester Utd": "Manchester United",
    "Man City": "Manchester City",
    "Newcastle Utd": "Newcastle United",
    "Newcastle": "Newcastle United",
    "Spurs": "Tottenham Hotspur",
    "Tottenham": "Tottenham Hotspur",
    "Wolves": "Wolverhampton Wanderers",
    "Wolverhampton": "Wolverhampton Wanderers",
    "West Brom": "West Bromwich Albion",
    "West Bromwich": "West Bromwich Albion",
    "Nott'm Forest": "Nottingham Forest",
    "Nottm Forest": "Nottingham Forest",
    "Sheffield Utd": "Sheffield United",
    "Sheff Utd": "Sheffield United",
    "Leicester": "Leicester City",
    "Norwich": "Norwich City",
    "Swansea": "Swansea City",
    "Stoke": "Stoke City",
    "Hull": "Hull City",
    "Cardiff": "Cardiff City",
    "QPR": "Queens Park Rangers",
    "Birmingham": "Birmingham City",
    "Bolton": "Bolton Wanderers",
    "Blackburn": "Blackburn Rovers",
    "Wigan": "Wigan Athletic",
    "West Ham": "West Ham United",
    "Leeds": "Leeds United",
    "Huddersfield": "Huddersfield Town",
    "Middlesboro": "Middlesbrough",
    "Charlton": "Charlton Athletic",
    "Portsmouth": "Portsmouth FC",
    "Brighton": "Brighton & Hove Albion",
    "Luton": "Luton Town",
}


def standardize_team_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Resolve inconsistent team-name variants to a single canonical name
    and strip stray whitespace introduced by scraping (e.g. 'Arsenal ').

    Without this step, the same club can fragment into multiple distinct
    entities downstream (e.g. in group-bys or standings calculations),
    silently corrupting any aggregate analysis.
    """
    df = df.copy()
    for col in ("HomeTeam", "AwayTeam"):
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()                 # trailing/leading whitespace
            .replace(TEAM_NAME_MAP)       # known-variant -> canonical name
        )
    return df


# ---------------------------------------------------------------------------
# PHASE 3: DOMAIN-SPECIFIC NULL HANDLING
# ---------------------------------------------------------------------------

def handle_goal_and_result_nulls(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply football-specific (NOT statistical) null-handling rules.

    - FTHG/FTAG missing -> the match has no recorded final score, which in
      practice means postponed/abandoned/unplayed. There is no valid way
      to "estimate" a football score, so these rows are dropped outright
      (imputing goals via mean/median would fabricate results).
    - FTR missing but goals present -> this is just a derivable label, not
      real missing information, so we deterministically recompute it from
      FTHG/FTAG instead of dropping the row.
    - HTHG/HTAG/HTR missing -> this is a genuine, structural gap for older
      seasons (half-time stats weren't tracked). We explicitly leave these
      as NaN rather than inferring half-time state from the full-time score.
    """
    df = df.copy()
    before = len(df)

    # --- Drop postponed/abandoned matches (no final score recorded) -----
    df = df.dropna(subset=["FTHG", "FTAG"])
    dropped_no_score = before - len(df)
    if dropped_no_score:
        print(f"  [DROP] {dropped_no_score} rows dropped: missing FTHG/FTAG "
              f"(postponed/abandoned matches, not statistically imputable).")

    # --- Deterministically recompute FTR where goals exist but FTR is NaN
    missing_ftr_mask = df["FTR"].isna()
    n_missing_ftr = missing_ftr_mask.sum()
    if n_missing_ftr:
        df.loc[missing_ftr_mask, "FTR"] = np.select(
            [
                df.loc[missing_ftr_mask, "FTHG"] > df.loc[missing_ftr_mask, "FTAG"],
                df.loc[missing_ftr_mask, "FTHG"] < df.loc[missing_ftr_mask, "FTAG"],
            ],
            ["H", "A"],
            default="D",
        )
        print(f"  [FIX] {n_missing_ftr} rows had missing FTR deterministically "
              f"re-derived from FTHG/FTAG (H/D/A logic).")

    # --- HTHG/HTAG/HTR: leave as structural NaN for older seasons --------
    # No action needed — explicitly NOT touching these columns is the point.
    for col in ("HTHG", "HTAG", "HTR"):
        if col not in df.columns:
            df[col] = np.nan  # ensure column exists even if no file had it

    return df


# ---------------------------------------------------------------------------
# PHASE 4: FOOTBALL SANITY CHECKS (DATA LEAKAGE / REALITY CHECKS)
# ---------------------------------------------------------------------------

def apply_sanity_checks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Enforce basic laws of football reality that scraped data can violate
    due to scraping/merging errors.

    - Time Travel Check: a team cannot have scored MORE goals by half-time
      than it had by full-time. HTHG > FTHG (or HTAG > FTAG) is logically
      impossible in a single match and indicates corrupted source data.
      Rows where HT values are NaN (structurally missing) correctly pass
      this check since NaN comparisons evaluate to False, not True.
    - Self-Play Check: a team cannot play itself. HomeTeam == AwayTeam
      indicates a scraping/parsing error and the row is dropped.
    """
    df = df.copy()

    # --- Time travel check (only applies where HT data actually exists) --
    time_travel_violation = (
        (df["HTHG"] > df["FTHG"]) | (df["HTAG"] > df["FTAG"])
    )
    n_time_travel = time_travel_violation.sum()
    if n_time_travel:
        print(f"  [DROP] {n_time_travel} rows dropped: half-time goals "
              f"exceeded full-time goals (impossible / corrupted data).")
    df = df.loc[~time_travel_violation]

    # --- Self-play check ---------------------------------------------------
    self_play_violation = df["HomeTeam"] == df["AwayTeam"]
    n_self_play = self_play_violation.sum()
    if n_self_play:
        print(f"  [DROP] {n_self_play} rows dropped: HomeTeam == AwayTeam "
              f"(a team cannot play itself).")
    df = df.loc[~self_play_violation]

    return df


# ---------------------------------------------------------------------------
# MAIN PIPELINE
# ---------------------------------------------------------------------------

def run_pipeline() -> pd.DataFrame:
    """Execute the full ingest -> clean -> validate pipeline end to end."""

    print("=" * 70)
    print("STEP 1/6: Upload & unify CSV files")
    print("=" * 70)
    df = upload_and_concat_csvs()

    print("\n" + "=" * 70)
    print("STEP 2/6: Parse match dates")
    print("=" * 70)
    df = parse_match_dates(df)

    print("\n" + "=" * 70)
    print("STEP 3/6: Assign season labels (Aug-May football calendar)")
    print("=" * 70)
    df = assign_season(df)

    print("\n" + "=" * 70)
    print("STEP 4/6: Standardize team names (entity resolution)")
    print("=" * 70)
    df = standardize_team_names(df)

    print("\n" + "=" * 70)
    print("STEP 5/6: Domain-specific null handling (goals / results)")
    print("=" * 70)
    df = handle_goal_and_result_nulls(df)

    print("\n" + "=" * 70)
    print("STEP 6/6: Football sanity checks (time-travel / self-play)")
    print("=" * 70)
    df = apply_sanity_checks(df)

    # Tidy final column order: core schema first, metadata last.
    preferred_order = [
        "Season", "Date", "HomeTeam", "AwayTeam",
        "FTHG", "FTAG", "FTR", "HTHG", "HTAG", "HTR",
    ]
    remaining_cols = [c for c in df.columns if c not in preferred_order]
    df = df[[c for c in preferred_order if c in df.columns] + remaining_cols]

    df = df.sort_values("Date").reset_index(drop=True)

    print("\n" + "=" * 70)
    print(f"PIPELINE COMPLETE: {len(df)} clean rows across "
          f"{df['Season'].nunique()} seasons.")
    print("=" * 70)

    return df


if __name__ == "__main__":
    clean_df = run_pipeline()
    display(clean_df.head(20))  # noqa: F821  (Colab provides `display`)

    # Optional: download the cleaned dataset as a single CSV.
    output_path = "epl_cleaned_unified.csv"
    clean_df.to_csv(output_path, index=False)
    files.download(output_path)

STEP 1/6: Upload & unify CSV files


Saving England Football Results.zip to England Football Results.zip
  [EXTRACTED] England Football Results.zip -> /content/epl_csv_extracted
  Found 158 CSV file(s) inside the zip.
  [LOADED] 1993-94_E0.csv: 552 rows, 29 cols
  [LOADED] 2000-01_E0.csv: 380 rows, 46 cols
  [LOADED] 2011-12_E0.csv: 380 rows, 72 cols
  [LOADED] 2000-01_E2.csv: 552 rows, 46 cols
  [LOADED] 2010-11_EC.csv: 552 rows, 72 cols
  [LOADED] 1993-94_E3.csv: 552 rows, 29 cols
  [LOADED] 2015-16_E0.csv: 380 rows, 66 cols
  [SKIPPED] Could not parse '/content/epl_csv_extracted/England Football Results/2004-05_E1.csv': 'utf-8' codec can't decode byte 0xa0 in position 116297: invalid start byte
  [SKIPPED] Could not parse '/content/epl_csv_extracted/England Football Results/2003-04_E0.csv': Error tokenizing data. C error: Expected 57 fields in line 305, saw 72

  [SKIPPED] Could not parse '/content/epl_csv_extracted/England Football Results/1993-94_E1.csv': Error tokenizing data. C error: Expected 28 fields in line 17,

/tmp/ipykernel_824/823277967.py:157: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")


  [WARN] 893 rows have an unparseable Date and will have no Season assigned (will be caught by goal-null drop logic later if also missing scores).

STEP 3/6: Assign season labels (Aug-May football calendar)

STEP 4/6: Standardize team names (entity resolution)

STEP 5/6: Domain-specific null handling (goals / results)
  [DROP] 893 rows dropped: missing FTHG/FTAG (postponed/abandoned matches, not statistically imputable).

STEP 6/6: Football sanity checks (time-travel / self-play)

PIPELINE COMPLETE: 67435 clean rows across 33 seasons.


,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Div,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,__source_file,Attendance,Referee,HS,AS,HST,AST,HHW,AHW,HC,AC,HF,AF,HO,AO,HY,AY,HR,AR,HBP,ABP,GBH,GBD,GBA,IWH,IWD,IWA,LBH,LBD,LBA,SBH,SBD,SBA,WHH,WHD,WHA,B365H,B365D,B365A,BWH,BWD,BWA,SJH,SJD,SJA,VCH,VCD,VCA,BSH,BSD,BSA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,PSH,PSD,PSA,PSCH,PSCD,PSCA,Time,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,B365>2.5,B365<2.5,P>2.5,P<2.5,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,AHh,B365AHH,B365AHA,PAHH,PAHA,MaxAHH,MaxAHA,AvgAHH,AvgAHA,B365CH,B365CD,B365CA,BWCH,BWCD,BWCA,IWCH,IWCD,IWCA,WHCH,WHCD,WHCA,VCCH,VCCD,VCCA,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,AvgCA,B365C>2.5,B365C<2.5,PC>2.5,PC<2.5,MaxC>2.5,MaxC<2.5,AvgC>2.5,AvgC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFDH,BFDD,BFDA,BMGMH,BMGMD,BMGMA,BVH,BVD,BVA,CLH,CLD,CLA,BFEH,BFED,BFEA,BFE>2.5,BFE<2.5,BFEAHH,BFEAHA,BFDCH,BFDCD,BFDCA,BMGMCH,BMGMCD,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA,BFECH,BFECD,BFECA,BFEC>2.5,BFEC<2.5,BFECAHH,BFECAHA,Unnamed: 132,BFH,BFD,BFA,1XBH,1XBD,1XBA,BFCH,BFCD,BFCA,1XBCH,1XBCD,1XBCA,SYH,SYD,SYA,PPH,PPD,PPA,SKBH,SKBD,SKBA,PPCH,PPCD,PPCA,SKBCH,SKBCD,SKBCA,SOH,SOD,SOA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,LBAHH,LBAHA,LBAH,B365AH,HxG,AxG,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52
0,1993/1994,1993-08-14,Arsenal,Coventry,0.0,3.0,A,NaN,NaN,NaN,E0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1993-94_E0.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1993/1994,1993-08-14,Wrexham,Rotherham,3.0,3.0,D,NaN,NaN,NaN,E2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1993-94_E2.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1993/1994,1993-08-14,Plymouth,Stockport,2.0,3.0,A,NaN,NaN,NaN,E2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1993-94_E2.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
LightGBM Model: EPL Match Result Classification (H / D / A)
==============================================================
Standalone modeling script. Assumes you already have the CLEANED,
unified EPL DataFrame (output of epl_data_cleaner.py) -- either as the
`clean_df` variable still in memory from that script, or loaded fresh
from the 'epl_cleaned_unified.csv' it downloads.

NOTE: Elo ratings are intentionally NOT included yet. Features here are
limited to rolling recent form + rest days. Elo can be added later as
its own feature block once this baseline is validated.
"""

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import (
    log_loss, accuracy_score, classification_report, confusion_matrix
)

pd.set_option("display.max_columns", None)


# ---------------------------------------------------------------------------
# LOAD CLEANED DATA (if not already in memory as `clean_df`)
# ---------------------------------------------------------------------------

def load_clean_data(path: str = "epl_cleaned_unified.csv") -> pd.DataFrame:
    """Load the cleaned/unified CSV produced by the data-cleaning pipeline."""
    df = pd.read_csv(path, parse_dates=["Date"])
    return df


# ---------------------------------------------------------------------------
# FEATURE ENGINEERING (PRE-MATCH, NO LEAKAGE) -- NO ELO YET
# ---------------------------------------------------------------------------

FORM_WINDOW = 5  # rolling window size (matches) for recent form


def compute_rolling_form(df: pd.DataFrame, window: int = FORM_WINDOW) -> pd.DataFrame:
    """
    For each team, compute rolling "form" over their last `window` matches
    (points earned, goals scored, goals conceded) -- regardless of whether
    those matches were home or away for them. Every rolling value is
    shifted by 1 so a match's own result never leaks into its own feature.

    Approach: reshape each match into two team-centric rows (one for the
    home side, one for the away side), compute rolling stats per team on
    that long table, then merge the home-perspective and away-perspective
    rows back onto the original match rows.
    """
    df = df.sort_values("Date").reset_index(drop=True)
    df["match_id"] = df.index

    # Home-perspective rows
    home_rows = pd.DataFrame({
        "match_id": df["match_id"],
        "Date": df["Date"],
        "team": df["HomeTeam"],
        "goals_for": df["FTHG"],
        "goals_against": df["FTAG"],
        "points": np.select(
            [df["FTR"] == "H", df["FTR"] == "D"], [3, 1], default=0
        ),
        "is_home": 1,
    })

    # Away-perspective rows
    away_rows = pd.DataFrame({
        "match_id": df["match_id"],
        "Date": df["Date"],
        "team": df["AwayTeam"],
        "goals_for": df["FTAG"],
        "goals_against": df["FTHG"],
        "points": np.select(
            [df["FTR"] == "A", df["FTR"] == "D"], [3, 1], default=0
        ),
        "is_home": 0,
    })

    long_df = pd.concat([home_rows, away_rows], ignore_index=True)
    long_df = long_df.sort_values(["team", "Date"])

    # shift(1) BEFORE rolling: the rolling window for a given match only
    # includes STRICTLY PRIOR matches, never the match itself.
    grp = long_df.groupby("team")
    long_df["form_pts"] = grp["points"].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).mean()
    )
    long_df["form_goals_for"] = grp["goals_for"].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).mean()
    )
    long_df["form_goals_against"] = grp["goals_against"].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).mean()
    )
    # Rest days: gap since this team's previous match (fixture congestion).
    long_df["rest_days"] = grp["Date"].transform(lambda s: s.diff().dt.days)

    # Split back into home-perspective and away-perspective feature sets
    # and merge each onto the original match rows via match_id.
    home_feats = long_df[long_df["is_home"] == 1][
        ["match_id", "form_pts", "form_goals_for", "form_goals_against", "rest_days"]
    ].rename(columns={
        "form_pts": "home_form_pts",
        "form_goals_for": "home_form_goals_for",
        "form_goals_against": "home_form_goals_against",
        "rest_days": "home_rest_days",
    })

    away_feats = long_df[long_df["is_home"] == 0][
        ["match_id", "form_pts", "form_goals_for", "form_goals_against", "rest_days"]
    ].rename(columns={
        "form_pts": "away_form_pts",
        "form_goals_for": "away_form_goals_for",
        "form_goals_against": "away_form_goals_against",
        "rest_days": "away_rest_days",
    })

    df = df.merge(home_feats, on="match_id", how="left")
    df = df.merge(away_feats, on="match_id", how="left")
    df = df.drop(columns=["match_id"])
    return df


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Orchestrate feature building: rolling form only (no Elo yet), plus
    the target label.

    A team's FIRST-EVER match in the dataset will have NaN rolling-form
    features (no history yet) -- LightGBM handles NaN splits natively,
    so these are deliberately left unimputed rather than filled with a
    mean/median (same "no statistical imputation of football reality"
    principle used in the cleaning pipeline).
    """
    df = compute_rolling_form(df)

    # Target: 0 = Away win, 1 = Draw, 2 = Home win (ordered so higher =
    # better for the home side, purely for readability of confusion matrices).
    result_map = {"A": 0, "D": 1, "H": 2}
    df["target"] = df["FTR"].map(result_map)

    return df


FEATURE_COLUMNS = [
    "HomeTeam", "AwayTeam",            # native categorical, LightGBM-native
    "home_form_pts", "away_form_pts",
    "home_form_goals_for", "home_form_goals_against",
    "away_form_goals_for", "away_form_goals_against",
    "home_rest_days", "away_rest_days",
]


# ---------------------------------------------------------------------------
# TRAIN / TEST SPLIT (TIME-BASED)
# ---------------------------------------------------------------------------

def time_based_split(df: pd.DataFrame, test_seasons: int = 2):
    """
    Split chronologically: the most recent `test_seasons` seasons become
    the held-out test set, everything earlier is training data. A random
    split would leak future squad/form patterns into training and give an
    unrealistically optimistic (and useless) evaluation.
    """
    seasons_sorted = sorted(df["Season"].dropna().unique())
    test_season_set = set(seasons_sorted[-test_seasons:])

    train_df = df[~df["Season"].isin(test_season_set)].copy()
    test_df = df[df["Season"].isin(test_season_set)].copy()

    print(f"  Train: {len(train_df)} matches "
          f"({seasons_sorted[0]} -> {seasons_sorted[-test_seasons - 1]})")
    print(f"  Test:  {len(test_df)} matches ({', '.join(sorted(test_season_set))})")

    return train_df, test_df


# ---------------------------------------------------------------------------
# LIGHTGBM TRAINING
# ---------------------------------------------------------------------------

def train_lightgbm(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    Train a multiclass LightGBM classifier on pre-match features to
    predict FTR (Away / Draw / Home). Rows with no usable target (NaN,
    shouldn't exist after cleaning) are dropped defensively.
    """
    train_df = train_df.dropna(subset=["target"])
    test_df = test_df.dropna(subset=["target"])

    # HomeTeam/AwayTeam as pandas 'category' dtype lets LightGBM split on
    # them natively (optimal grouping of categories) instead of needing
    # one-hot encoding across 40+ historical club names.
    cat_features = ["HomeTeam", "AwayTeam"]
    for col in cat_features:
        train_df[col] = train_df[col].astype("category")
        test_df[col] = test_df[col].astype(
            pd.CategoricalDtype(categories=train_df[col].cat.categories)
        )

    X_train, y_train = train_df[FEATURE_COLUMNS], train_df["target"].astype(int)
    X_test, y_test = test_df[FEATURE_COLUMNS], test_df["target"].astype(int)

    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=3,
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=30,     # guards against overfitting on a modest
                                   # dataset size (a few thousand matches)
        class_weight="balanced",  # draws are a structural minority class
        random_state=42,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="multi_logloss",
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
    )

    return model, X_test, y_test


# ---------------------------------------------------------------------------
# EVALUATION
# ---------------------------------------------------------------------------

def evaluate_model(model, X_test, y_test):
    """
    Report multiclass log-loss (primary metric -- penalizes confident
    wrong predictions, unlike accuracy) alongside accuracy, a full
    classification report, and the confusion matrix.
    """
    proba = model.predict_proba(X_test)
    preds = np.argmax(proba, axis=1)

    label_names = ["Away Win", "Draw", "Home Win"]

    print(f"  Log Loss:  {log_loss(y_test, proba):.4f}")
    print(f"  Accuracy:  {accuracy_score(y_test, preds):.4f}")
    print("\n  Classification Report:")
    print(classification_report(y_test, preds, target_names=label_names))

    print("  Confusion Matrix (rows=actual, cols=predicted):")
    cm = confusion_matrix(y_test, preds)
    print(pd.DataFrame(cm, index=label_names, columns=label_names))

    # Feature importance -- sanity check on which pre-match signals the
    # model is actually leaning on.
    importance = pd.Series(
        model.feature_importances_, index=FEATURE_COLUMNS
    ).sort_values(ascending=False)
    print("\n  Feature Importances:")
    print(importance)

    return proba, preds


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

def run_modeling(clean_df: pd.DataFrame = None):
    """
    Full modeling flow: load (if needed) -> build features -> split ->
    train -> evaluate. Pass in `clean_df` directly if it's already in
    memory from the cleaning script; otherwise it's loaded from disk.
    """
    if clean_df is None:
        print("Loading cleaned data from disk...")
        clean_df = load_clean_data()

    print("\n" + "=" * 70)
    print("STEP 1/3: Build pre-match features (rolling form, no Elo)")
    print("=" * 70)
    feat_df = build_features(clean_df)

    print("\n" + "=" * 70)
    print("STEP 2/3: Time-based train/test split")
    print("=" * 70)
    train_df, test_df = time_based_split(feat_df, test_seasons=2)

    print("\n" + "=" * 70)
    print("STEP 3/3: Train LightGBM + evaluate")
    print("=" * 70)
    model, X_test, y_test = train_lightgbm(train_df, test_df)
    evaluate_model(model, X_test, y_test)

    return model, feat_df


# ---------------------------------------------------------------------------
# PREDICTING A NEW / UPCOMING FIXTURE
# ---------------------------------------------------------------------------
#
# You do NOT hand-type form/rest-day numbers. Those are derived from each
# team's most recent matches in feat_df. You only supply which two teams
# are playing (and optionally the match date, for an accurate rest-days
# calculation). This function looks up each team's latest known rolling
# form and assembles exactly the one-row feature frame the model expects.

def _latest_team_snapshot(feat_df: pd.DataFrame, team: str, as_of_date=None):
    """
    Return the most recent pre-match form snapshot available for `team`,
    using matches strictly before `as_of_date` (defaults to "now" --
    i.e. every match already in feat_df). Looks at both the team's
    HOME rows and AWAY rows and takes whichever is more recent, since
    "recent form" should reflect the team's last match regardless of
    venue.
    """
    if as_of_date is not None:
        as_of_date = pd.Timestamp(as_of_date)
        df = feat_df[feat_df["Date"] < as_of_date]
    else:
        df = feat_df

    home_side = df[df["HomeTeam"] == team][
        ["Date", "home_form_pts", "home_form_goals_for",
         "home_form_goals_against", "home_rest_days"]
    ].rename(columns=lambda c: c.replace("home_", ""))

    away_side = df[df["AwayTeam"] == team][
        ["Date", "away_form_pts", "away_form_goals_for",
         "away_form_goals_against", "away_rest_days"]
    ].rename(columns=lambda c: c.replace("away_", ""))

    combined = pd.concat([home_side, away_side]).sort_values("Date")

    if combined.empty:
        # Team has no history at all (e.g. newly promoted club) --
        # leave as NaN rather than inventing a value. LightGBM handles it.
        return {"form_pts": np.nan, "form_goals_for": np.nan,
                "form_goals_against": np.nan, "last_match_date": None}

    latest = combined.iloc[-1]
    return {
        "form_pts": latest["form_pts"],
        "form_goals_for": latest["form_goals_for"],
        "form_goals_against": latest["form_goals_against"],
        "last_match_date": latest["Date"],
    }


def predict_fixture(model, feat_df: pd.DataFrame, home_team: str,
                     away_team: str, match_date=None):
    """
    Predict H/D/A probabilities for one upcoming fixture.

    Inputs you actually provide:
      - home_team, away_team: the two club names (must match the
        standardized names used in TEAM_NAME_MAP from the cleaning script,
        e.g. "Manchester United", not "Man Utd").
      - match_date (optional): date of the fixture, used only to compute
        each team's rest days since their last match. Defaults to today.

    Everything else (recent form, goals for/against) is looked up
    automatically from feat_df -- the same historical table the model
    was trained on.
    """
    match_date = pd.Timestamp(match_date) if match_date else pd.Timestamp.now()

    home_snap = _latest_team_snapshot(feat_df, home_team, as_of_date=match_date)
    away_snap = _latest_team_snapshot(feat_df, away_team, as_of_date=match_date)

    def _rest_days(snap):
        if snap["last_match_date"] is None:
            return np.nan
        return (match_date - snap["last_match_date"]).days

    row = pd.DataFrame([{
        "HomeTeam": home_team,
        "AwayTeam": away_team,
        "home_form_pts": home_snap["form_pts"],
        "away_form_pts": away_snap["form_pts"],
        "home_form_goals_for": home_snap["form_goals_for"],
        "home_form_goals_against": home_snap["form_goals_against"],
        "away_form_goals_for": away_snap["form_goals_for"],
        "away_form_goals_against": away_snap["form_goals_against"],
        "home_rest_days": _rest_days(home_snap),
        "away_rest_days": _rest_days(away_snap),
    }])

    # Match the categorical dtype/levels the model was trained on, so
    # unseen-but-known team names map correctly and brand-new names
    # become NaN-category rather than crashing. We use the full set of
    # team names seen anywhere in feat_df (home or away columns) as the
    # category universe, same as what training used.
    all_teams = pd.unique(
        pd.concat([feat_df["HomeTeam"], feat_df["AwayTeam"]])
    )
    for col in ("HomeTeam", "AwayTeam"):
        row[col] = row[col].astype(pd.CategoricalDtype(categories=all_teams))

    proba = model.predict_proba(row[FEATURE_COLUMNS])[0]
    label_names = ["Away Win", "Draw", "Home Win"]

    result = dict(zip(label_names, proba))
    print(f"  {home_team} (H) vs {away_team} (A) on "
          f"{match_date.date()}:")
    for label, p in result.items():
        print(f"    {label:10s}: {p:.1%}")

    return result


if __name__ == "__main__":
    # If `clean_df` already exists in the notebook namespace (from running
    # epl_data_cleaner.py in the same session), it'll be picked up here.
    try:
        clean_df  # noqa: F821
    except NameError:
        clean_df = None

    lgb_model, feature_df = run_modeling(clean_df)

    # Example: predict an upcoming fixture. Swap in real team names/date.
    predict_fixture(
        lgb_model, feature_df,
        home_team="Manchester United",
        away_team="Liverpool",
        match_date="2026-11-01",
    )


STEP 1/3: Build pre-match features (rolling form, no Elo)

STEP 2/3: Time-based train/test split
  Train: 64427 matches (1993/1994 -> 2024/2025)
  Test:  3008 matches (2025/2026, 2026/2027)

STEP 3/3: Train LightGBM + evaluate
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009234 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 700
[LightGBM] [Info] Number of data points in the train set: 64427, number of used features: 10
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
  Log Loss:  1.0876
  Accuracy:  0.3853

  Classification Report:
              precision    recall  f1-score   support

    Away Win       0.36      0.41      0.38       934
        Draw       0.28      0.32      0.30       777
    Home Win       0.49      0.41      0.45      1297

    accuracy                          